# 3b. LSTM Temporal Modeling on DCNN FC7 Features (Soft-Label Aware)

This notebook trains an LSTM on FC7 features extracted from a pretrained DCNN and now supports both hard-label and soft-label segment supervision.

## Quick Summary of Current Pipeline

1. Load segment arrays and utterance ids from either `original` or `loso` split style.
2. Extract FC7 features for each segment using a DCNN checkpoint.
3. Build utterance-level variable-length sequences.
4. Train a bidirectional LSTM with:
   - hard-label cross entropy (`LABEL_MODE="hard"`), or
   - soft-target cross entropy (`LABEL_MODE="soft"`).
5. Convert per-segment probabilities to top-2 distributions (renormalized to sum to 1).
6. Apply temporal window aggregation (`left 2 + current + right 2`) and assign the dominant emotion to each segment.
7. Evaluate with:
   - Section A1: single-emotion segment classification report,
   - Section A2: two-emotion transition pair table (ordered 7x6) with MSE and top-2 set match,
   - Section B: interval-level t-IoU metrics by emotion.

## Section 1. Importing Libraries and Making Runs Reproducible

### What is happening here?
This section imports all required libraries for sequence modeling, soft-label evaluation, temporal interval metrics, and reporting. It also fixes random seeds for reproducibility.

### Why this matters
The notebook includes both hard and soft supervision paths, plus window-based probability smoothing and pair-wise transition analysis. Stable seeds make comparisons across settings more reliable.

### Main items used here
- `numpy`, `torch`, and `torchvision` for feature/model pipeline
- `sklearn.metrics` for segment classification summaries
- `defaultdict`, `Counter` for grouping and temporal voting
- `EMOTION_ENG_MAP`, `EMOTION_CODE_MAP` for readable reporting and timeline parsing

### Step-by-step flow
1. Import core scientific and deep learning packages.
2. Import project-specific emotion mappings.
3. Set deterministic seed behavior.
4. Keep runtime behavior consistent across reruns.

In [ ]:
import os
import csv
from collections import defaultdict, Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, classification_report

from utility import EMOTION_ENG_MAP, EMOTION_CODE_MAP

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Section 2. Global Settings and Experiment Choices

### What is happening here?
This section is the experiment control panel. It now includes switches for data style and label style, so the same notebook supports classic hard-label training and combined-data soft-label training.

### Important parameters
- `DATASET`: folder containing preprocessed arrays
- `DATA_KIND`: `"original"` or `"combined"`
- `LABEL_MODE`: `"hard"` or `"soft"`
- `SPLIT_MODE`: `"original"` or `"loso"`
- `NUM_CLASSES = 7`: emotion class count
- `SMOOTH_RADIUS = 2`: temporal window radius (left2/right2)
- `TIOU_THRESHOLD = 0.50`: interval success threshold

### Step-by-step flow
1. Build data/model/results paths.
2. Set timing constants for interval conversion.
3. Discover folds when using LOSO.
4. Print all core mode settings before training.

In [ ]:
CURR_DIR = os.getcwd()

#===================================================

# Update as needed
DATASET = "processed_emodb_comb_norm_loso_soft"
DATA_KIND = "combined"  # "original" or "combined"
LABEL_MODE = "soft"     # "hard" or "soft"
SPLIT_MODE = "loso"  # "original" or "loso"

#===================================================

DATASET_PATH = os.path.join(CURR_DIR, DATASET)
MODEL_DIR = os.path.join(CURR_DIR, "models")
RESULTS_DIR = os.path.join(CURR_DIR, "results", DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)

RAW_TIMELINE_ROOT = os.path.abspath(os.path.join(CURR_DIR, "../emo_db_comb"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7
HOP_MS = 0.010
WINDOW_MS = 0.025
SEG_FRAMES = 64
FRAME_SHIFT = 30
SEG_HOP_SEC = HOP_MS * FRAME_SHIFT
SEG_DUR_SEC = WINDOW_MS + (SEG_FRAMES - 1) * HOP_MS

SMOOTH_WINDOW = 5
SMOOTH_RADIUS = 2  # left 2 + right 2 + current segment
TIOU_THRESHOLD = 0.50

if SPLIT_MODE == "loso":
    FOLD_NAMES = sorted([
        name for name in os.listdir(DATASET_PATH)
        if name.startswith("fold") and os.path.isdir(os.path.join(DATASET_PATH, name))
    ])
    if len(FOLD_NAMES) == 0:
        raise ValueError(f"No fold directories found under {DATASET_PATH}.")
else:
    FOLD_NAMES = []

print(f"DATASET_PATH: {DATASET_PATH}")
print(f"DATA_KIND: {DATA_KIND}")
print(f"LABEL_MODE: {LABEL_MODE}")
print(f"SPLIT_MODE: {SPLIT_MODE}")
if SPLIT_MODE == "loso":
    print(f"Folds: {FOLD_NAMES}")

## Section 3. Turning Each Audio Segment into a Deep Feature

### What is happening here?
A pretrained AlexNet-style DCNN checkpoint is converted into an FC7 feature extractor so each segment is represented by a compact 4096-dimensional vector.

### Why this matters
The LSTM learns temporal emotion patterns from FC7 sequences rather than raw spectrogram tensors, reducing learning complexity and improving sequence modeling stability.

### Important parameters
- Input segment shape: `3 x 64 x 64`
- AlexNet resized input: `227 x 227`
- Output feature size: `4096`
- Batch extraction size: `64`

### Step-by-step flow
1. Define transform pipeline.
2. Rebuild classifier head to match class count and load checkpoint.
3. Remove final classifier layer to expose FC7.
4. Extract FC7 features batch-wise for each utterance segment list.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((227, 227), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def build_fc7_extractor(model_path):
    model = models.alexnet()
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.classifier = nn.Sequential(*list(model.classifier.children())[:5])
    model.eval()
    return model.to(device)

def preprocess_segment_for_alexnet(seg):
    img = seg.copy()
    for c in range(3):
        mn, mx = img[c].min(), img[c].max()
        img[c] = (img[c] - mn) / (mx - mn) if mx > mn else 0.0
    img = img.transpose(1, 2, 0)
    return transform(img).float()

def extract_fc7_batch(segments, extractor, batch_size=64):
    tensors = torch.stack([preprocess_segment_for_alexnet(s) for s in segments], dim=0)
    outputs = []
    with torch.no_grad():
        for i in range(0, len(tensors), batch_size):
            b = tensors[i:i+batch_size].to(device=device, dtype=torch.float32)
            outputs.append(extractor(b).cpu())
    return torch.cat(outputs, dim=0).numpy()

## Section 4. Grouping Segments into Utterance Sequences

### What is happening here?
This section groups segment-level inputs into utterance-level sequences and now carries both hard and soft supervision metadata through the data loader.

### Why this matters
For transition-aware training/reporting, each segment may include: hard label, soft label vector, active emotion count, and ordered pair metadata. All of these are needed later for split reporting.

### Important parameters
- Inputs: `X_*.npy`, `y_*.npy`, `utterance_ids_*.npy`
- Optional soft targets: `y_soft_*.npy`
- Optional pair metadata: `y_pair_*.npy`
- Padding hard-label value: `-100`

### Step-by-step flow
1. Load split arrays.
2. Load soft/pair arrays when `LABEL_MODE="soft"`.
3. Compute active-emotion counts from soft labels.
4. Group by utterance id and extract FC7 per utterance.
5. Pad batched sequences for features, hard labels, soft labels, active counts, and pair metadata.

In [ ]:
def group_segments_by_utterance(X, y_hard, y_soft, y_pair, active_count, utt_ids):
    grouped = defaultdict(lambda: {
        "segments": [],
        "labels_hard": [],
        "labels_soft": [],
        "active_count": [],
        "pair": [],
    })
    for i, uid in enumerate(utt_ids):
        key = str(uid)
        grouped[key]["segments"].append(X[i])
        grouped[key]["labels_hard"].append(int(y_hard[i]))
        grouped[key]["labels_soft"].append(y_soft[i])
        grouped[key]["active_count"].append(int(active_count[i]))
        grouped[key]["pair"].append(y_pair[i])
    return grouped

def build_sequences_from_split(dataset_dir, split_name, extractor):
    X = np.load(os.path.join(dataset_dir, f"X_{split_name}.npy"))
    y_hard = np.load(os.path.join(dataset_dir, f"y_{split_name}.npy")).astype(np.int64)
    utt_ids = np.load(os.path.join(dataset_dir, f"utterance_ids_{split_name}.npy"))

    soft_path = os.path.join(dataset_dir, f"y_soft_{split_name}.npy")
    pair_path = os.path.join(dataset_dir, f"y_pair_{split_name}.npy")

    if LABEL_MODE == "soft" and os.path.exists(soft_path):
        y_soft = np.load(soft_path).astype(np.float32)
    else:
        y_soft = np.eye(NUM_CLASSES, dtype=np.float32)[y_hard]

    if LABEL_MODE == "soft" and os.path.exists(pair_path):
        y_pair = np.load(pair_path).astype(np.int64)
    else:
        y_pair = np.full((len(y_hard), 2), fill_value=-1, dtype=np.int64)

    active_count = (y_soft > 1e-8).sum(axis=1).astype(np.int64)

    grouped = group_segments_by_utterance(X, y_hard, y_soft, y_pair, active_count, utt_ids)
    seq_feats, seq_labels_hard, seq_labels_soft = [], [], []
    seq_active_count, seq_pair, seq_uids = [], [], []

    for uid, pack in grouped.items():
        seg_arr = np.array(pack["segments"])
        fc7 = extract_fc7_batch(seg_arr, extractor)
        seq_feats.append(torch.tensor(fc7, dtype=torch.float32))
        seq_labels_hard.append(torch.tensor(pack["labels_hard"], dtype=torch.long))
        seq_labels_soft.append(torch.tensor(np.array(pack["labels_soft"], dtype=np.float32), dtype=torch.float32))
        seq_active_count.append(torch.tensor(pack["active_count"], dtype=torch.long))
        seq_pair.append(torch.tensor(np.array(pack["pair"], dtype=np.int64), dtype=torch.long))
        seq_uids.append(uid)

    return seq_feats, seq_labels_hard, seq_labels_soft, seq_active_count, seq_pair, seq_uids

class SequenceDataset(Dataset):
    def __init__(self, feats, labels_hard, labels_soft, active_count, pair, uids):
        self.feats = feats
        self.labels_hard = labels_hard
        self.labels_soft = labels_soft
        self.active_count = active_count
        self.pair = pair
        self.uids = uids

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        return (
            self.feats[idx],
            self.labels_hard[idx],
            self.labels_soft[idx],
            self.active_count[idx],
            self.pair[idx],
            self.uids[idx],
        )

def collate_sequences(batch):
    feats, labels_hard, labels_soft, active_count, pair, uids = zip(*batch)
    lengths = torch.tensor([len(x) for x in feats], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = feats[0].shape[1]

    pad_feats = torch.zeros(len(batch), max_len, feat_dim, dtype=torch.float32)
    pad_labels_hard = torch.full((len(batch), max_len), fill_value=-100, dtype=torch.long)
    pad_labels_soft = torch.zeros(len(batch), max_len, NUM_CLASSES, dtype=torch.float32)
    pad_active_count = torch.zeros(len(batch), max_len, dtype=torch.long)
    pad_pair = torch.full((len(batch), max_len, 2), fill_value=-1, dtype=torch.long)

    for i, (f, lh, ls, ac, pr) in enumerate(zip(feats, labels_hard, labels_soft, active_count, pair)):
        T = len(f)
        pad_feats[i, :T] = f
        pad_labels_hard[i, :T] = lh
        pad_labels_soft[i, :T] = ls
        pad_active_count[i, :T] = ac
        pad_pair[i, :T] = pr

    return pad_feats, pad_labels_hard, pad_labels_soft, pad_active_count, pad_pair, lengths, list(uids)

## Section 5. LSTM Model and Top-2 Temporal Smoothing

### What is happening here?
This section defines the bidirectional LSTM and the updated temporal post-processing strategy used for combined/soft-label evaluation.

### Updated smoothing behavior
Instead of majority-vote smoothing on hard labels, the notebook now:
1. Converts each segment probability vector to top-2 only.
2. Renormalizes top-2 values so they sum to 1.
3. Applies a temporal window (`left 2 + current + right 2`) by summing probabilities across the window.
4. Assigns the dominant emotion (argmax summed probability) to the current segment.

### Why this matters
This preserves uncertainty between two likely emotions while still producing stable segment decisions for interval conversion and t-IoU evaluation.

### Important parameters
- LSTM input size: `4096`
- Hidden size: `256`
- Bidirectional outputs
- Smoothing radius: `SMOOTH_RADIUS = 2`

In [ ]:
class LSTMTagger(nn.Module):
    def __init__(self, input_dim=4096, hidden_dim=256, num_layers=1, num_classes=7, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        logits = self.head(out)
        return logits

def smooth_labels(seq, window=5):
    if window <= 1 or len(seq) == 0:
        return list(seq)
    half = window // 2
    out = []
    for i in range(len(seq)):
        left = max(0, i - half)
        right = min(len(seq), i + half + 1)
        votes = Counter(seq[left:right])
        out.append(votes.most_common(1)[0][0])
    return out

def keep_top2_and_renorm(prob_vec):
    top2_idx = np.argsort(prob_vec)[-2:]
    out = np.zeros_like(prob_vec, dtype=np.float32)
    out[top2_idx] = prob_vec[top2_idx]
    denom = float(out.sum())
    if denom > 0.0:
        out /= denom
    return out

def window_smooth_from_probs(prob_seq, radius=2):
    if len(prob_seq) == 0:
        return [], np.zeros((0, NUM_CLASSES), dtype=np.float32)
    smoothed_labels = []
    smoothed_prob = []
    arr = np.array(prob_seq, dtype=np.float32)
    n = arr.shape[0]
    for i in range(n):
        left = max(0, i - radius)
        right = min(n, i + radius + 1)
        sum_prob = arr[left:right].sum(axis=0)
        best = int(np.argmax(sum_prob))
        smoothed_labels.append(best)
        smoothed_prob.append(sum_prob / max(sum_prob.sum(), 1e-12))
    return smoothed_labels, np.array(smoothed_prob, dtype=np.float32)

def sequence_to_intervals(labels, seg_hop_sec=SEG_HOP_SEC, seg_dur_sec=SEG_DUR_SEC):
    if len(labels) == 0:
        return []
    intervals = []
    start_idx = 0
    curr = labels[0]
    for i in range(1, len(labels)):
        if labels[i] != curr:
            s = start_idx * seg_hop_sec
            e = (i - 1) * seg_hop_sec + seg_dur_sec
            intervals.append((s, e, curr))
            start_idx = i
            curr = labels[i]
    s = start_idx * seg_hop_sec
    e = (len(labels) - 1) * seg_hop_sec + seg_dur_sec
    intervals.append((s, e, curr))
    return intervals

def interval_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return (inter / union) if union > 0 else 0.0

def dominant_label(intervals):
    if not intervals:
        return 0
    dur = defaultdict(float)
    for s, e, l in intervals:
        dur[int(l)] += max(0.0, e - s)
    return sorted(dur.items(), key=lambda x: (-x[1], x[0]))[0][0]

## Section 6. Loading Timeline Labels and Measuring t-IoU

### What is happening here?
This section loads timeline CSV annotations and evaluates temporal overlap quality between predicted intervals and ground-truth intervals.

### Why this matters
For multi-emotion and transition-heavy utterances, segment accuracy alone is insufficient. t-IoU evaluates whether the model places emotions in the correct time regions.

### Matching rule
- Compare intervals only when emotion labels match.
- For each ground-truth interval, keep the best-overlap IoU from predicted intervals.
- Report mean best-overlap IoU and success rate at `TIOU_THRESHOLD`.

### Step-by-step flow
1. Parse speaker timeline CSV files.
2. Build lookup by speaker/utterance id.
3. Compute interval IoU overlaps.
4. Aggregate mean t-IoU and success statistics.

In [ ]:
def load_timeline_lookup(root_dir):
    lookup = defaultdict(dict)
    if not os.path.isdir(root_dir):
        return lookup

    for name in sorted(os.listdir(root_dir)):
        if not name.startswith("speaker_"):
            continue
        spk = name.split("_")[-1]
        csv_path = os.path.join(root_dir, name, f"speaker_{spk}_timeline.csv")
        if not os.path.exists(csv_path):
            continue

        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                uid = row.get("generated_file", "").strip()
                code = row.get("emotion_code", "").strip()
                if uid == "" or code not in EMOTION_CODE_MAP:
                    continue
                s = float(row.get("start_sec", 0.0))
                e = float(row.get("end_sec", 0.0))
                lookup[spk].setdefault(uid, []).append((s, e, EMOTION_CODE_MAP[code]))

    for spk in lookup:
        for uid in lookup[spk]:
            lookup[spk][uid].sort(key=lambda x: x[0])
    return lookup

def compute_interval_successes(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD):
    rows = []
    for gs, ge, gl in gt_intervals:
        best_iou = 0.0
        for ps, pe, pl in pred_intervals:
            if int(pl) != int(gl):
                continue
            best_iou = max(best_iou, interval_iou(gs, ge, ps, pe))
        rows.append({
            "label": int(gl),
            "best_iou": float(best_iou),
            "success": bool(best_iou >= threshold),
        })
    return rows

def compute_tiou(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD):
    if len(gt_intervals) == 0:
        return 0.0, 0.0, []

    rows = compute_interval_successes(pred_intervals, gt_intervals, threshold=threshold)
    best_ious = np.array([r["best_iou"] for r in rows], dtype=np.float32)
    mean_tiou = float(best_ious.mean()) if len(best_ious) > 0 else 0.0
    success_rate = float((best_ious >= threshold).mean()) if len(best_ious) > 0 else 0.0
    return mean_tiou, success_rate, rows

## Section 7. Training the LSTM and Collecting Evaluation Results

### What is happening here?
This section runs training and evaluation for one split/fold and supports both hard-label and soft-label supervision.

### Training modes
- `LABEL_MODE="hard"`: cross-entropy on integer class labels
- `LABEL_MODE="soft"`: soft-target cross-entropy on probability labels

### Evaluation outputs collected
- Segment-level hard predictions (after top-2 window smoothing)
- Segment-level soft predictions (top-2 renormalized probabilities)
- Ground-truth hard/soft labels
- Active emotion count and pair metadata
- Interval-level t-IoU statistics by emotion

### Why this matters
These outputs enable split reporting for single-emotion vs transition segments and keep interval-level temporal evaluation unchanged.

In [ ]:
def soft_target_cross_entropy(logits, soft_targets):
    log_probs = torch.log_softmax(logits, dim=1)
    return -(soft_targets * log_probs).sum(dim=1).mean()

def train_lstm_one_run(train_loader, val_loader, max_epochs=20, lr=1e-3, hidden_dim=256):
    model = LSTMTagger(hidden_dim=hidden_dim, num_classes=NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    best_val = float('inf')
    best_state = None
    patience = 5
    bad = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        train_losses = []
        for feats, labels_hard, labels_soft, _, _, _, _ in train_loader:
            feats = feats.to(device)
            labels_hard = labels_hard.to(device)
            labels_soft = labels_soft.to(device)
            logits = model(feats)

            valid_mask = labels_hard != -100
            if LABEL_MODE == "soft":
                logits_valid = logits[valid_mask]
                soft_valid = labels_soft[valid_mask]
                loss = soft_target_cross_entropy(logits_valid, soft_valid) if logits_valid.numel() > 0 else logits.sum() * 0.0
            else:
                loss = criterion(logits.reshape(-1, NUM_CLASSES), labels_hard.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for feats, labels_hard, labels_soft, _, _, _, _ in val_loader:
                feats = feats.to(device)
                labels_hard = labels_hard.to(device)
                labels_soft = labels_soft.to(device)
                logits = model(feats)

                valid_mask = labels_hard != -100
                if LABEL_MODE == "soft":
                    logits_valid = logits[valid_mask]
                    soft_valid = labels_soft[valid_mask]
                    val_loss = soft_target_cross_entropy(logits_valid, soft_valid) if logits_valid.numel() > 0 else logits.sum() * 0.0
                else:
                    val_loss = criterion(logits.reshape(-1, NUM_CLASSES), labels_hard.reshape(-1))
                val_losses.append(val_loss.item())

        tr = float(np.mean(train_losses)) if train_losses else 0.0
        va = float(np.mean(val_losses)) if val_losses else 0.0
        history.append({
            "epoch": epoch + 1,
            "train_loss": tr,
            "val_loss": va,
        })
        print(f"Epoch {epoch+1:02d} | train_loss={tr:.4f} | val_loss={va:.4f}")

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

def evaluate_lstm_sequences(model, loader, timeline_lookup):
    model.eval()
    all_tiou = []
    all_success = []
    seg_true_hard_all = []
    seg_true_soft_all = []
    seg_pred_hard_all = []
    seg_pred_soft_all = []
    seg_active_count_all = []
    seg_pair_all = []
    interval_total = np.zeros(NUM_CLASSES, dtype=np.int64)
    interval_success = np.zeros(NUM_CLASSES, dtype=np.int64)
    interval_iou_sum = np.zeros(NUM_CLASSES, dtype=np.float64)
    ce_criterion = nn.CrossEntropyLoss(ignore_index=-100, reduction='sum')
    ce_sum = 0.0
    ce_tokens = 0

    with torch.no_grad():
        for feats, labels_hard, labels_soft, active_count, pair, lengths, uids in loader:
            feats = feats.to(device)
            labels_dev = labels_hard.to(device)
            logits_t = model(feats)

            loss_sum = ce_criterion(logits_t.reshape(-1, NUM_CLASSES), labels_dev.reshape(-1))
            valid_tokens = int((labels_dev != -100).sum().item())
            ce_sum += float(loss_sum.item())
            ce_tokens += valid_tokens

            probs_np = torch.softmax(logits_t, dim=-1).cpu().numpy()
            labels_hard_np = labels_hard.numpy()
            labels_soft_np = labels_soft.numpy()
            active_count_np = active_count.numpy()
            pair_np = pair.numpy()
            lengths_np = lengths.numpy()

            for b in range(len(uids)):
                T = int(lengths_np[b])
                uid = str(uids[b])

                pred_top2 = np.array([keep_top2_and_renorm(v) for v in probs_np[b, :T]], dtype=np.float32)
                pred_smoothed_hard, _ = window_smooth_from_probs(pred_top2, radius=SMOOTH_RADIUS)
                pred_intervals = sequence_to_intervals(pred_smoothed_hard)

                true_soft_seq = labels_soft_np[b, :T]
                true_hard_seq = labels_hard_np[b, :T].astype(np.int64).tolist()

                spk = uid[:2]
                gt_intervals = timeline_lookup.get(spk, {}).get(uid, None)
                if gt_intervals is None:
                    gt_intervals = sequence_to_intervals(true_hard_seq)

                m_tiou, s_rate, tiou_rows = compute_tiou(pred_intervals, gt_intervals, threshold=TIOU_THRESHOLD)
                all_tiou.append(m_tiou)
                all_success.append(s_rate)

                for r in tiou_rows:
                    emo = int(r["label"])
                    interval_total[emo] += 1
                    interval_iou_sum[emo] += float(r["best_iou"])
                    if r["success"]:
                        interval_success[emo] += 1

                seg_true_hard_all.extend(true_hard_seq)
                seg_true_soft_all.extend(true_soft_seq.tolist())
                seg_pred_hard_all.extend(pred_smoothed_hard)
                seg_pred_soft_all.extend(pred_top2.tolist())
                seg_active_count_all.extend(active_count_np[b, :T].astype(np.int64).tolist())
                seg_pair_all.extend(pair_np[b, :T].astype(np.int64).tolist())

    seg_true_hard_np = np.array(seg_true_hard_all, dtype=np.int64)
    seg_pred_hard_np = np.array(seg_pred_hard_all, dtype=np.int64)
    seg_true_soft_np = np.array(seg_true_soft_all, dtype=np.float32)
    seg_pred_soft_np = np.array(seg_pred_soft_all, dtype=np.float32)
    seg_active_count_np = np.array(seg_active_count_all, dtype=np.int64)
    seg_pair_np = np.array(seg_pair_all, dtype=np.int64)
    seg_acc = accuracy_score(seg_true_hard_np, seg_pred_hard_np) if len(seg_true_hard_np) > 0 else 0.0
    cross_entropy = float(ce_sum / ce_tokens) if ce_tokens > 0 else 0.0

    return {
        "y_true_seg": seg_true_hard_np,
        "y_pred_seg": seg_pred_hard_np,
        "y_true_soft_seg": seg_true_soft_np,
        "y_pred_soft_seg": seg_pred_soft_np,
        "active_count_seg": seg_active_count_np,
        "pair_seg": seg_pair_np,
        "segment_accuracy": float(seg_acc),
        "num_segments": int(len(seg_true_hard_np)),
        "mean_tiou": float(np.mean(all_tiou)) if all_tiou else 0.0,
        "success_rate": float(np.mean(all_success)) if all_success else 0.0,
        "cross_entropy": cross_entropy,
        "ce_sum": float(ce_sum),
        "ce_tokens": int(ce_tokens),
        "interval_total": interval_total,
        "interval_success": interval_success,
        "interval_iou_sum": interval_iou_sum,
    }

## Section 8. Running the Full Experiment and Writing the Final Report

### What is happening here?
This section loops through the selected split/folds, trains and evaluates the model, and writes a consolidated report with soft-label-aware segment diagnostics plus interval metrics.

### Final report structure
- **Section A1**: Single-emotion segment classification report
- **Section A2**: Ordered two-emotion transition pair table (7x6)
  - count per pair
  - MSE between predicted and target soft vectors
  - top-2 set match rate
- **Section B**: Interval-level temporal detection by emotion (`tIoU`, `tIoU@thr`, `Succ`, `Total`)

### Why this matters
The report now separates discrete single-label quality from continuous transition quality, while preserving temporal overlap evaluation through t-IoU.

In [ ]:
timeline_lookup = load_timeline_lookup(RAW_TIMELINE_ROOT)

all_seg_true = []
all_seg_pred = []
all_seg_true_soft = []
all_seg_pred_soft = []
all_seg_active_count = []
all_seg_pair = []
sum_interval_total = np.zeros(NUM_CLASSES, dtype=np.int64)
sum_interval_success = np.zeros(NUM_CLASSES, dtype=np.int64)
sum_interval_iou = np.zeros(NUM_CLASSES, dtype=np.float64)
fold_tious = []
fold_success = []
fold_seg_acc = []
fold_cross_entropy = []
fold_training_logs = []
total_ce_sum = 0.0
total_ce_tokens = 0

if SPLIT_MODE == "original":
    dcnn_path = os.path.join(MODEL_DIR, f"dcnn_{DATASET}.pth")
    if not os.path.exists(dcnn_path):
        raise FileNotFoundError(f"Missing DCNN checkpoint: {dcnn_path}")

    extractor = build_fc7_extractor(dcnn_path)
    tr_f, tr_lh, tr_ls, tr_ac, tr_pr, tr_u = build_sequences_from_split(DATASET_PATH, "train", extractor)
    va_f, va_lh, va_ls, va_ac, va_pr, va_u = build_sequences_from_split(DATASET_PATH, "validation", extractor)
    te_f, te_lh, te_ls, te_ac, te_pr, te_u = build_sequences_from_split(DATASET_PATH, "test", extractor)

    train_loader = DataLoader(SequenceDataset(tr_f, tr_lh, tr_ls, tr_ac, tr_pr, tr_u), batch_size=16, shuffle=True, collate_fn=collate_sequences)
    val_loader = DataLoader(SequenceDataset(va_f, va_lh, va_ls, va_ac, va_pr, va_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)
    test_loader = DataLoader(SequenceDataset(te_f, te_lh, te_ls, te_ac, te_pr, te_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)

    model, history = train_lstm_one_run(train_loader, val_loader)
    eval_out = evaluate_lstm_sequences(model, test_loader, timeline_lookup)

    fold_name = "original"
    fold_training_logs.append((fold_name, history))
    fold_tious.append(eval_out["mean_tiou"])
    fold_success.append(eval_out["success_rate"])
    fold_seg_acc.append(eval_out["segment_accuracy"])
    fold_cross_entropy.append(eval_out["cross_entropy"])
    total_ce_sum += eval_out["ce_sum"]
    total_ce_tokens += eval_out["ce_tokens"]

    all_seg_true.extend(eval_out["y_true_seg"].tolist())
    all_seg_pred.extend(eval_out["y_pred_seg"].tolist())
    all_seg_true_soft.extend(eval_out["y_true_soft_seg"].tolist())
    all_seg_pred_soft.extend(eval_out["y_pred_soft_seg"].tolist())
    all_seg_active_count.extend(eval_out["active_count_seg"].tolist())
    all_seg_pair.extend(eval_out["pair_seg"].tolist())
    sum_interval_total += eval_out["interval_total"]
    sum_interval_success += eval_out["interval_success"]
    sum_interval_iou += eval_out["interval_iou_sum"]

    print(f"Segment accuracy: {eval_out['segment_accuracy'] * 100:.2f}%")
    print(f"Cross-entropy (test): {eval_out['cross_entropy']:.4f}")
    print(f"Mean t-IoU: {eval_out['mean_tiou']:.4f}")
    print(f"t-IoU success@{TIOU_THRESHOLD:.2f}: {eval_out['success_rate'] * 100:.2f}%")

else:
    for fold_name in FOLD_NAMES:
        print('=' * 80)
        print(f"Fold: {fold_name}")
        fold_path = os.path.join(DATASET_PATH, fold_name)
        dcnn_path = os.path.join(MODEL_DIR, f"dcnn_{DATASET}_{fold_name}.pth")
        if not os.path.exists(dcnn_path):
            raise FileNotFoundError(f"Missing DCNN checkpoint: {dcnn_path}")

        extractor = build_fc7_extractor(dcnn_path)
        tr_f, tr_lh, tr_ls, tr_ac, tr_pr, tr_u = build_sequences_from_split(fold_path, "train", extractor)
        va_f, va_lh, va_ls, va_ac, va_pr, va_u = build_sequences_from_split(fold_path, "validation", extractor)
        te_f, te_lh, te_ls, te_ac, te_pr, te_u = build_sequences_from_split(fold_path, "test", extractor)

        train_loader = DataLoader(SequenceDataset(tr_f, tr_lh, tr_ls, tr_ac, tr_pr, tr_u), batch_size=16, shuffle=True, collate_fn=collate_sequences)
        val_loader = DataLoader(SequenceDataset(va_f, va_lh, va_ls, va_ac, va_pr, va_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)
        test_loader = DataLoader(SequenceDataset(te_f, te_lh, te_ls, te_ac, te_pr, te_u), batch_size=16, shuffle=False, collate_fn=collate_sequences)

        model, history = train_lstm_one_run(train_loader, val_loader)
        eval_out = evaluate_lstm_sequences(model, test_loader, timeline_lookup)

        fold_training_logs.append((fold_name, history))
        fold_tious.append(eval_out["mean_tiou"])
        fold_success.append(eval_out["success_rate"])
        fold_seg_acc.append(eval_out["segment_accuracy"])
        fold_cross_entropy.append(eval_out["cross_entropy"])
        total_ce_sum += eval_out["ce_sum"]
        total_ce_tokens += eval_out["ce_tokens"]

        all_seg_true.extend(eval_out["y_true_seg"].tolist())
        all_seg_pred.extend(eval_out["y_pred_seg"].tolist())
        all_seg_true_soft.extend(eval_out["y_true_soft_seg"].tolist())
        all_seg_pred_soft.extend(eval_out["y_pred_soft_seg"].tolist())
        all_seg_active_count.extend(eval_out["active_count_seg"].tolist())
        all_seg_pair.extend(eval_out["pair_seg"].tolist())
        sum_interval_total += eval_out["interval_total"]
        sum_interval_success += eval_out["interval_success"]
        sum_interval_iou += eval_out["interval_iou_sum"]

        print(f"Fold segment accuracy: {eval_out['segment_accuracy'] * 100:.2f}%")
        print(f"Fold cross-entropy (test): {eval_out['cross_entropy']:.4f}")
        print(f"Fold mean t-IoU: {eval_out['mean_tiou']:.4f}")
        print(f"Fold t-IoU success@{TIOU_THRESHOLD:.2f}: {eval_out['success_rate'] * 100:.2f}%")

In [ ]:
y_true_seg_final = np.array(all_seg_true, dtype=np.int64)
y_pred_seg_final = np.array(all_seg_pred, dtype=np.int64)
y_true_soft_final = np.array(all_seg_true_soft, dtype=np.float32)
y_pred_soft_final = np.array(all_seg_pred_soft, dtype=np.float32)
active_count_final = np.array(all_seg_active_count, dtype=np.int64)
pair_final = np.array(all_seg_pair, dtype=np.int64) if len(all_seg_pair) > 0 else np.empty((0, 2), dtype=np.int64)

global_seg_acc = accuracy_score(y_true_seg_final, y_pred_seg_final) if len(y_true_seg_final) > 0 else 0.0
emotion_names = [EMOTION_ENG_MAP.get(i, f"Class_{i}") for i in range(NUM_CLASSES)]

single_mask = active_count_final == 1
switch_mask = active_count_final == 2

single_acc = accuracy_score(y_true_seg_final[single_mask], y_pred_seg_final[single_mask]) if single_mask.any() else 0.0
single_report = (
    classification_report(
        y_true_seg_final[single_mask],
        y_pred_seg_final[single_mask],
        labels=np.arange(NUM_CLASSES),
        target_names=emotion_names,
        digits=4,
        zero_division=0,
    )
    if single_mask.any() else "No single-emotion segments to report."
)

def infer_pair_from_soft(soft_vec):
    idx = np.argsort(-soft_vec)[:2]
    if len(idx) < 2 or int(idx[0]) == int(idx[1]):
        return (-1, -1)
    return (int(idx[0]), int(idx[1]))

pair_stats = {}
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j:
            pair_stats[(i, j)] = {"count": 0, "mse_sum": 0.0, "top2_set_match": 0}

for idx in np.where(switch_mask)[0]:
    pair_key = tuple(int(x) for x in pair_final[idx]) if idx < len(pair_final) else (-1, -1)
    if pair_key[0] < 0 or pair_key[1] < 0 or pair_key[0] == pair_key[1]:
        pair_key = infer_pair_from_soft(y_true_soft_final[idx])
    if pair_key[0] < 0 or pair_key[1] < 0 or pair_key[0] == pair_key[1]:
        continue

    pred_soft = y_pred_soft_final[idx]
    true_soft = y_true_soft_final[idx]
    mse = float(np.mean((pred_soft - true_soft) ** 2))
    pred_top2_set = set(np.argsort(pred_soft)[-2:].tolist())
    true_top2_set = set(np.argsort(true_soft)[-2:].tolist())

    pair_stats[pair_key]["count"] += 1
    pair_stats[pair_key]["mse_sum"] += mse
    if pred_top2_set == true_top2_set:
        pair_stats[pair_key]["top2_set_match"] += 1

pair_rows = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i == j:
            continue
        s = pair_stats[(i, j)]
        c = s["count"]
        pair_rows.append({
            "pair": f"{emotion_names[i]} -> {emotion_names[j]}",
            "count": c,
            "mse": (s["mse_sum"] / c) if c > 0 else 0.0,
            "top2_match": (s["top2_set_match"] / c) if c > 0 else 0.0,
        })

final_tiou = float(np.mean(fold_tious)) if fold_tious else 0.0
final_tiou_success = float(np.mean(fold_success)) if fold_success else 0.0
final_seg_acc_mean = float(np.mean(fold_seg_acc)) if fold_seg_acc else 0.0
final_ce_mean = float(np.mean(fold_cross_entropy)) if fold_cross_entropy else 0.0
global_ce = float(total_ce_sum / total_ce_tokens) if total_ce_tokens > 0 else 0.0
total_intervals_counted = int(sum_interval_total.sum())

emotion_rows = []
for emo in range(NUM_CLASSES):
    name = EMOTION_ENG_MAP.get(emo, str(emo))
    interval_success_val = int(sum_interval_success[emo])
    interval_total_val = int(sum_interval_total[emo])
    tiou_mean_val = float(sum_interval_iou[emo] / interval_total_val) if interval_total_val > 0 else 0.0
    tiou_thr_val = float(interval_success_val / interval_total_val) if interval_total_val > 0 else 0.0
    emotion_rows.append({
        "emotion_id": emo,
        "emotion": name,
        "tiou": tiou_mean_val,
        "tiou_thr": tiou_thr_val,
        "interval_success": interval_success_val,
        "interval_total": interval_total_val,
    })

table_header = f"{'Emotion':<12} {'tIoU':>10} {'tIoU@thr':>10} {'Succ':>8} {'Total':>8}"
table_sep = '-' * len(table_header)
pair_header = f"{'Pair':<24} {'Count':>8} {'MSE':>12} {'Top2Set':>10}"
pair_sep = '-' * len(pair_header)

print('=' * 80)
print(f"Final global per-segment accuracy ({SPLIT_MODE}): {global_seg_acc * 100:.2f}%")
print(f"Single-emotion segment accuracy               : {single_acc * 100:.2f}%")
print(f"Final mean fold per-segment accuracy          : {final_seg_acc_mean * 100:.2f}%")
print(f"Final cross-entropy (global token-weighted)   : {global_ce:.4f}")
print(f"Final cross-entropy (mean across folds)       : {final_ce_mean:.4f}")
print(f"Final mean t-IoU                              : {final_tiou:.4f}")
print(f"Final t-IoU success@{TIOU_THRESHOLD:.2f}      : {final_tiou_success * 100:.2f}%")
print(f"Final Section B total GT intervals counted    : {total_intervals_counted}")

print("\nSection A1. Single-emotion segment classification:")
print(single_report)

print("Section A2. Two-emotion transition pairs (ordered, 7x6):")
print(pair_header)
print(pair_sep)
for row in pair_rows:
    print(f"{row['pair']:<24} {row['count']:>8d} {row['mse']:>12.5f} {row['top2_match']:>10.3f}")

print("\nSection B. Interval-level temporal detection by emotion (aggregated over all test utterances):")
print(table_header)
print(table_sep)
for row in emotion_rows:
    print(
        f"{row['emotion']:<12} "
        f"{row['tiou']:>10.3f} {row['tiou_thr']:>10.3f} "
        f"{row['interval_success']:>8d} {row['interval_total']:>8d}"
    )

print("Interpretation:")
print("    - tIoU     : mean best-overlap IoU per ground-truth interval for that emotion.")
print("    - tIoU@thr : fraction of intervals with IoU >= threshold for that emotion.")
print("    - Succ     : count of successful intervals for that emotion.")
print("    - Total    : count of all ground-truth intervals for that emotion.")

report_path = os.path.join(RESULTS_DIR, f"classification_report_lstm_{DATASET}.txt")

with open(report_path, "w") as f:
    f.write(f"{f'Dataset                                ':<38} : {DATASET}\n")
    f.write(f"{f'Data kind                              ':<38} : {DATA_KIND}\n")
    f.write(f"{f'Label mode                             ':<38} : {LABEL_MODE}\n")
    f.write(f"{f'Split mode                             ':<38} : {SPLIT_MODE}\n")
    f.write(f"{f'Global Mean per-segment accuracy       ':<38} : {global_seg_acc * 100:.2f}%\n")
    f.write(f"{f'Single-emotion segment accuracy        ':<38} : {single_acc * 100:.2f}%\n")
    f.write(f"{f'Fold Mean per-segment accuracy         ':<38} : {final_seg_acc_mean * 100:.2f}%\n")
    f.write(f"{f'Global Mean Cross-entropy              ':<38} : {global_ce:.4f}\n")
    f.write(f"{f'Fold Mean Cross-entropy                ':<38} : {final_ce_mean:.4f}\n")
    f.write(f"{f't-IoU (Fold Mean)                      ':<38} : {final_tiou:.4f}\n")
    f.write(f"{f't-IoU success@{TIOU_THRESHOLD:.2f}     ':<38} : {final_tiou_success * 100:.2f}%\n")
    f.write(f"{f'Section B total GT intervals counted   ':<38} : {total_intervals_counted}\n\n")

    f.write("Note:\n")
    f.write("  Global metrics are weighted by fold size while fold metrics are unweighted across folds.\n")
    f.write("  They match only when all folds have the same number of segments.\n")

    f.write("\nClass index mapping:\n")
    for idx, name in enumerate(emotion_names):
        f.write(f"{idx}: {name}\n")

    f.write("\nSection A1. Single-emotion segment classification report:\n")
    f.write(single_report)

    f.write("\nSection A2. Two-emotion transition pairs (ordered, 7x6):\n")
    f.write(pair_header + "\n")
    f.write(pair_sep + "\n")
    for row in pair_rows:
        f.write(f"{row['pair']:<24} {row['count']:>8d} {row['mse']:>12.5f} {row['top2_match']:>10.3f}\n")

    f.write("\nSection B. Interval-level temporal detection by emotion (aggregated over all test utterances):\n")
    f.write(table_header + "\n")
    f.write(table_sep + "\n")
    for row in emotion_rows:
        f.write(
            f"{row['emotion']:<12} "
            f"{row['tiou']:>10.3f} {row['tiou_thr']:>10.3f} "
            f"{row['interval_success']:>8d} {row['interval_total']:>8d}\n"
        )

    f.write("\nInterpretation:\n")
    f.write("    - tIoU     : mean best-overlap IoU per ground-truth interval for that emotion.\n")
    f.write("    - tIoU@thr : fraction of intervals with IoU >= threshold for that emotion.\n")
    f.write("    - Succ     : count of successful intervals for that emotion.\n")
    f.write("    - Total    : count of all ground-truth intervals for that emotion.\n")

    f.write("\nTraining progress per split/fold (epoch-wise):\n")
    for fold_name, history in fold_training_logs:
        f.write(f"\n[{fold_name}]\n")
        for row in history:
            f.write(
                f"Epoch {row['epoch']:02d} | train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f}\n"
            )

print(f"Saved LSTM report to: {report_path}")